# Qwen3.5-4B Socratic tutor: baseline and QLoRA fine-tuning

This notebook measures how the pretrained Qwen3.5-4B behaves as a maths tutor, then fine-tunes it
with LoRA on the candidate dialogue set and measures it again on the same questions.

Steps:
1. Load `candidate_train` and `candidate_val` from the data-prep notebook.
2. Format each dialogue with the tutoring system prompt and mark which tutor turns are training targets.
3. Baseline: generate the next tutor turn for 200 validation contexts with the 4-bit model (and, optionally, the 16-bit model) and score the replies.
4. Hyperparameter search: train a small grid of LoRA settings on a subset and compare validation loss and reply metrics.
5. Final run: train the best setting on the full candidate set and repeat the baseline evaluation.

Run it on Kaggle with a T4 GPU and internet switched on. Add the `processed/` folder from the data-prep notebook as a Kaggle dataset.
The candidate train set is about 1 million tokens per epoch and each grid setting trains on about 230 thousand.
T4 speed for this model has not been measured yet: check `train_minutes` after the first grid setting, and shrink `GRID` or `GRID_TRAIN` if the full run would not fit in a Kaggle session.
Set `SMOKE_TEST = True` to run every step on CPU with a tiny random model; this checks the code, not the results.

Metrics, computed on the same evaluation contexts for every model:

| Metric | Meaning | Better |
|---|---|---|
| leak rate | reply states the final answer before the student has said it | lower |
| question rate | reply contains a question | higher for a Socratic tutor |
| ROUGE-L, BLEU | word overlap with the real tutor's next turn | higher |
| words | reply length | close to the real tutors |
| validation loss | masked next-token loss on the target tutor turns of `candidate_val` | lower |

In [ ]:
SMOKE_TEST = False          # True: tiny random model on CPU, a handful of examples (code check only)
FAST_KERNELS = True         # Triton kernels for Qwen3.5's linear-attention layers; set False if they fail on the T4

import os, subprocess, sys
if not SMOKE_TEST:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                           "transformers>=5.0,<6", "peft", "accelerate", "bitsandbytes", "rouge_score", "sacrebleu"])
    if FAST_KERNELS:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "flash-linear-attention"])
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # Kaggle offers two T4s; this notebook uses one
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation between runs

In [ ]:
import gc, glob, gzip, json, math, random, re, time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from rouge_score import rouge_scorer
import sacrebleu
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID, REVISION = "Qwen/Qwen3.5-4B", "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
LOAD_IN_4BIT = True               # 4-bit NF4 base model (QLoRA)
COMPARE_16BIT_BASELINE = True     # also baseline the fp16 model on the same contexts
MAX_LEN = 1024                    # tokens per training dialogue; longer dialogues are cut at a tutor turn (fits a T4)
EVAL_POINTS = 200                 # one evaluation context per validation dialogue
MAX_NEW_TOKENS = 128
GEN_BATCH = 8
SEED = 42

# LoRA grid: every combination is trained on GRID_TRAIN dialogues for one epoch
GRID = [dict(r=r, lora_alpha=r, learning_rate=lr) for r in (8, 16) for lr in (1e-4, 2e-4)]
GRID_TRAIN, GRID_VAL, GRID_EVAL_POINTS = 400, 100, 60
FINAL_EPOCHS = 1
COMMON = dict(lora_dropout=0.05, batch_size=1, grad_accum=8, warmup_ratio=0.05,
              target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

if SMOKE_TEST:
    LOAD_IN_4BIT, COMPARE_16BIT_BASELINE = False, False
    MAX_LEN, EVAL_POINTS, MAX_NEW_TOKENS, GEN_BATCH = 512, 4, 8, 2
    GRID, GRID_TRAIN, GRID_VAL, GRID_EVAL_POINTS = GRID[:2], 8, 4, 2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = Path("/kaggle/working/qwen35_socratic_ft") if Path("/kaggle/working").exists() else Path("ft_output")
(OUT / "figures").mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
CACHE_FILE = OUT / "run_cache.json"   # finished evaluations and grid settings; a rerun skips them
if not CACHE_FILE.exists():   # resume from a cache uploaded as a Kaggle input, if one exists
    previous = glob.glob("/kaggle/input/**/run_cache.json", recursive=True)
    if previous:
        CACHE_FILE.write_text(Path(previous[0]).read_text())
        print("resuming from", previous[0])
CACHE = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {"eval": {}, "grid": {}}

def save_cache():
    CACHE_FILE.write_text(json.dumps(CACHE, indent=1, default=str))

RESULTS = {"config": dict(model=MODEL_ID, revision=REVISION, load_in_4bit=LOAD_IN_4BIT, max_len=MAX_LEN,
                          eval_points=EVAL_POINTS, max_new_tokens=MAX_NEW_TOKENS, grid=GRID, common=COMMON,
                          grid_train=GRID_TRAIN, final_epochs=FINAL_EPOCHS, smoke_test=SMOKE_TEST)}

## 1. Data

In [ ]:
def find_file(name):
    # Kaggle may unpack .gz uploads, so accept either form
    for pattern in [f"/kaggle/input/**/{name}.jsonl*", f"../socratic_data_prep/processed/{name}.jsonl*", f"processed/{name}.jsonl*"]:
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(f"{name}.jsonl(.gz) not found; add the data-prep processed files as a Kaggle dataset")

def read_jsonl(path):
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

TRAIN_FILE, VAL_FILE = find_file("candidate_train"), find_file("candidate_val")
train_rows, val_rows = read_jsonl(TRAIN_FILE), read_jsonl(VAL_FILE)
if SMOKE_TEST:
    train_rows, val_rows = train_rows[:GRID_TRAIN], val_rows[:EVAL_POINTS]
print(TRAIN_FILE, "| train", Counter(r["source"] for r in train_rows), "| val", Counter(r["source"] for r in val_rows))

## 2. Prompt format and training targets

Every dialogue starts with the same system prompt: the tutoring instruction, the problem, the
correct final answer (to check the student against, never to reveal) and, for MathDial, the
student's first attempt. Tutor turns are the assistant role and student turns are the user role.
The tutor's turns become training targets unless MathDial labels them "telling" or they reveal the
final answer before the student has said it. Those turns stay in the context but get no loss.

Qwen3.5 adds an empty thinking block before the final assistant turn when thinking is switched off.
`render()` reproduces the official chat template exactly so that loss masks can be set by character
offsets; the assertion below checks it against `apply_chat_template`.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
tok.pad_token = tok.pad_token or tok.eos_token

SYSTEM = ("You are a Socratic maths tutor. Guide the student one step at a time with questions and hints. "
          "Never state the final answer.\n\nProblem: {question}\n"
          "Correct final answer (for checking only, never reveal it): {answer}")

def chat_messages(rec, upto=None):
    sys_msg = SYSTEM.format(question=rec["question"], answer=rec["reference_answer"])
    if rec.get("student_attempt"):
        sys_msg += "\n\nStudent's first attempt:\n" + rec["student_attempt"]
    msgs = rec["messages"] if upto is None else rec["messages"][:upto]
    return [{"role": "system", "content": sys_msg}] + [
        {"role": "assistant" if m["role"] == "tutor" else "user", "content": m["content"]} for m in msgs]

THINK_OFF = "<think>\n\n</think>\n\n"

def render(rec, n_messages):
    # Text identical to apply_chat_template(..., enable_thinking=False) plus character spans of each tutor turn
    msgs = chat_messages(rec, n_messages)
    last_user = max((i for i, m in enumerate(msgs) if m["role"] == "user"), default=-1)
    text, spans = "", []
    for i, m in enumerate(msgs):
        head = f"<|im_start|>{m['role']}\n" + (THINK_OFF if m["role"] == "assistant" and i > last_user else "")
        text += head
        start = len(text)
        text += m["content"] + "<|im_end|>"
        if m["role"] == "assistant":
            src = rec["messages"][i - 1]
            spans.append((start, len(text), src.get("move") != "telling" and not src.get("leaks_answer", False)))
        text += "\n"
    return text, spans

for rec in train_rows[:50]:
    n = len(rec["messages"])
    assert render(rec, n)[0] == tok.apply_chat_template(chat_messages(rec, n), tokenize=False, enable_thinking=False)
print("render() matches the official chat template")

def encode(rec):
    # Drop messages from the end until the dialogue fits MAX_LEN and still ends on a tutor turn
    n = len(rec["messages"])
    while n > 0:
        if rec["messages"][n - 1]["role"] == "tutor":
            text, spans = render(rec, n)
            enc = tok(text, return_offsets_mapping=True, add_special_tokens=False)
            if len(enc["input_ids"]) <= MAX_LEN:
                break
        n -= 1
    if n == 0:
        return None
    labels = [-100] * len(enc["input_ids"])
    for k, (a, b) in enumerate(enc["offset_mapping"]):
        if any(train and a >= s and b <= e for s, e, train in spans):
            labels[k] = enc["input_ids"][k]
    if all(l == -100 for l in labels):
        return None
    return dict(input_ids=enc["input_ids"], labels=labels, cut=n < len(rec["messages"]))

train_enc = [e for e in map(encode, train_rows) if e]
val_enc = [e for e in map(encode, val_rows) if e]
lens = [len(e["input_ids"]) for e in train_enc]
target_share = np.mean([sum(l != -100 for l in e["labels"]) / len(e["labels"]) for e in train_enc])
RESULTS["data"] = dict(train_examples=len(train_enc), val_examples=len(val_enc), train_tokens=int(sum(lens)),
                       median_tokens=int(np.median(lens)), max_tokens=int(max(lens)), cut=sum(e["cut"] for e in train_enc),
                       target_token_share=round(float(target_share), 3))
print(RESULTS["data"])

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(lens, bins=40, color="#4C72B0")
ax.set(title="Training dialogues: tokens (Qwen3.5 tokenizer)", xlabel="tokens", ylabel="dialogues")
fig.tight_layout(); fig.savefig(OUT / "figures/01_token_lengths.png"); plt.show()

## 3. Evaluation contexts

From each validation dialogue the notebook picks one tutor turn that follows a student message.
The model sees everything before that turn and writes the next tutor message. The real tutor's
message is the reference. The same contexts are used for the baseline, the grid and the final model.

In [ ]:
NUM = re.compile(r"(?<![\d./])-?\d[\d,]*(?:\.\d+)?(?!\d|\.\d|/)")

def numbers(text):
    return {n.replace(",", "").rstrip("0").rstrip(".") if "." in n else n.replace(",", "") for n in NUM.findall(text)}

def answer_key(rec):
    found = numbers(rec["reference_answer"])
    key = found.pop() if len(found) == 1 else None
    return None if key in numbers(rec["question"]) else key

rng = random.Random(SEED)
eval_points = []
for rec in val_rows:
    idx = [i for i, m in enumerate(rec["messages"]) if m["role"] == "tutor" and i > 0 and rec["messages"][i - 1]["role"] == "student"]
    if not idx:
        continue
    i = rng.choice(idx)
    key = answer_key(rec)
    said = key is not None and any(key in numbers(m["content"]) for m in rec["messages"][:i] if m["role"] == "student")
    eval_points.append(dict(dialogue_id=rec["dialogue_id"], source=rec["source"], key=key, student_said=said,
                            prompt=tok.apply_chat_template(chat_messages(rec, i), tokenize=False, add_generation_prompt=True, enable_thinking=False),
                            reference=rec["messages"][i]["content"]))
eval_points = eval_points[:EVAL_POINTS]
print(len(eval_points), Counter(p["source"] for p in eval_points))

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def score(replies, points):
    rows = []
    for reply, p in zip(replies, points):
        rows.append(dict(source=p["source"],
                         leak=p["key"] is not None and not p["student_said"] and p["key"] in numbers(reply),
                         question="?" in reply, words=len(reply.split()),
                         rougeL=scorer.score(p["reference"], reply)["rougeL"].fmeasure))
    df = pd.DataFrame(rows)
    out = dict(leak_rate=df.leak.mean(), question_rate=df.question.mean(), words=df.words.mean(), rougeL=df.rougeL.mean(),
               bleu=sacrebleu.corpus_bleu(replies, [[p["reference"] for p in points]]).score / 100)
    return {k: round(float(v), 4) for k, v in out.items()}

human = score([p["reference"] for p in eval_points], eval_points)
human.update(rougeL=1.0, bleu=1.0)
RESULTS["real_tutor_reference"] = human
print("real tutors on the same contexts:", human)

## 4. Model loading, generation and validation loss

In [ ]:
def tiny_model():
    # Random 4-layer Qwen3.5 text model with the real vocabulary, for SMOKE_TEST only
    from transformers import Qwen3_5TextConfig, Qwen3_5ForCausalLM
    cfg = Qwen3_5TextConfig(vocab_size=len(tok), hidden_size=64, intermediate_size=128, num_hidden_layers=4,
                            layer_types=["linear_attention"] * 3 + ["full_attention"], num_attention_heads=4,
                            num_key_value_heads=2, head_dim=16, linear_num_key_heads=2, linear_num_value_heads=4,
                            linear_key_head_dim=16, linear_value_head_dim=16, tie_word_embeddings=True)
    return Qwen3_5ForCausalLM(cfg)

def load_model(four_bit):
    if SMOKE_TEST:
        return tiny_model().to(DEVICE)
    kwargs = dict(revision=REVISION, dtype=torch.float16, device_map={"": 0})
    if four_bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)

def free():
    # call after `del model` so the GPU memory can actually be released
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

@torch.no_grad()
def generate(model, points):
    model.eval()
    tok.padding_side = "left"
    replies = []
    for i in range(0, len(points), GEN_BATCH):
        batch = tok([p["prompt"] for p in points[i:i + GEN_BATCH]], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        out = model.generate(**batch, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id)
        for seq in out[:, batch["input_ids"].shape[1]:]:
            text = tok.decode(seq, skip_special_tokens=True)
            replies.append(re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip())
    tok.padding_side = "right"
    return replies

@torch.no_grad()
def val_loss(model, examples):
    model.eval()
    total, count = 0.0, 0
    for e in examples:
        ids = torch.tensor([e["input_ids"]], device=DEVICE)
        labels = torch.tensor([e["labels"]], device=DEVICE)
        n = int((labels[:, 1:] != -100).sum())
        total += float(model(input_ids=ids, labels=labels).loss) * n
        count += n
    return round(total / count, 4)

def evaluate(model, name, points=None, examples=None):
    if name in CACHE["eval"]:
        print(name, "(cached)", CACHE["eval"][name])
        return CACHE["eval"][name]
    points, examples = points or eval_points, examples or val_enc
    t = time.time()
    replies = generate(model, points)
    metrics = score(replies, points)
    metrics["val_loss"] = val_loss(model, examples)
    metrics["eval_minutes"] = round((time.time() - t) / 60, 2)
    if any(not r for r in replies):
        metrics["empty_replies"] = sum(not r for r in replies)
    pd.DataFrame(dict(dialogue_id=[p["dialogue_id"] for p in points], source=[p["source"] for p in points],
                      reference=[p["reference"] for p in points], reply=replies)).to_csv(OUT / f"generations_{name}.csv", index=False)
    print(name, metrics)
    CACHE["eval"][name] = metrics
    save_cache()
    return metrics

## 5. Baseline (pretrained weights, no training)

The pretrained model is scored on the evaluation contexts, with the same prompt and greedy decoding
used later for the tuned model. Validation loss uses the same targets as training.

In [ ]:
base = load_model(LOAD_IN_4BIT)
RESULTS["baseline_4bit" if LOAD_IN_4BIT else "baseline"] = evaluate(base, "baseline_4bit" if LOAD_IN_4BIT else "baseline")
del base; free()

if COMPARE_16BIT_BASELINE:
    base16 = load_model(False)
    RESULTS["baseline_fp16"] = evaluate(base16, "baseline_fp16")
    del base16; free()

pd.DataFrame({k: v for k, v in RESULTS.items() if k.startswith("baseline") or k == "real_tutor_reference"}).T

## 6. Hyperparameter search

Search strategy: a grid over LoRA rank (8, 16) and learning rate (1e-4, 2e-4), with `lora_alpha`
equal to the rank. Each setting trains for one epoch on `GRID_TRAIN` dialogues and is scored on
`GRID_VAL` validation dialogues (loss) and `GRID_EVAL_POINTS` evaluation contexts (reply metrics).
The rest stays fixed: dropout 0.05, effective batch 8, cosine schedule, 5% warm-up, 8-bit paged AdamW,
gradient checkpointing.

Selection rule: lowest validation loss among the settings whose leak rate is no higher than the
baseline's. If none qualifies, the lowest validation loss overall.

In [ ]:
def collate(batch):
    width = max(len(e["input_ids"]) for e in batch)
    pad = lambda seq, v: seq + [v] * (width - len(seq))
    return dict(input_ids=torch.tensor([pad(e["input_ids"], tok.pad_token_id) for e in batch]),
                labels=torch.tensor([pad(e["labels"], -100) for e in batch]),
                attention_mask=torch.tensor([pad([1] * len(e["input_ids"]), 0) for e in batch]))

class ListDataset(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

# One base model for every run. Reloading it after each training run leaked a full copy (about 2.4 GiB)
# on the T4, so each setting gets a fresh LoRA adapter that is removed again with unload().
BASE = load_model(LOAD_IN_4BIT)
if LOAD_IN_4BIT:
    BASE = prepare_model_for_kbit_training(BASE, use_gradient_checkpointing=True)
elif DEVICE == "cuda":
    BASE.gradient_checkpointing_enable()

def release_adapter(model):
    # Remove the LoRA layers and return the clean base model for the next run
    base = model.unload()
    if hasattr(base, "peft_config"):
        del base.peft_config
    return base

def train_lora(setting, examples, epochs, run_name):
    torch.manual_seed(SEED)   # same adapter initialisation for every setting
    model = get_peft_model(BASE, LoraConfig(r=setting["r"], lora_alpha=setting["lora_alpha"], lora_dropout=COMMON["lora_dropout"],
                                             target_modules=COMMON["target_modules"], bias="none", task_type="CAUSAL_LM"))
    args = TrainingArguments(output_dir=str(OUT / "checkpoints" / run_name), num_train_epochs=epochs,
                             per_device_train_batch_size=COMMON["batch_size"], gradient_accumulation_steps=COMMON["grad_accum"],
                             learning_rate=setting["learning_rate"], lr_scheduler_type="cosine", warmup_steps=COMMON["warmup_ratio"],
                             logging_steps=1 if SMOKE_TEST else 5, save_strategy="no", report_to="none", seed=SEED,
                             fp16=DEVICE == "cuda", optim="paged_adamw_8bit" if DEVICE == "cuda" else "adamw_torch",
                             gradient_checkpointing=DEVICE == "cuda", remove_unused_columns=False,
                             max_steps=2 if SMOKE_TEST else -1)
    t = time.time()
    trainer = Trainer(model=model, args=args, train_dataset=ListDataset(examples), data_collator=collate)
    trainer.train()
    history = [h for h in trainer.state.log_history if "loss" in h]
    model = trainer.model
    trainer.accelerator.free_memory()   # drop the optimizer state and the trainer's references to the model
    del trainer; free()
    return model, dict(train_minutes=round((time.time() - t) / 60, 2), history=history,
                       trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad))

baseline_name = "baseline_4bit" if LOAD_IN_4BIT else "baseline"
grid_train = random.Random(SEED).sample(train_enc, min(GRID_TRAIN, len(train_enc)))
grid_val, grid_points = val_enc[:GRID_VAL], eval_points[:GRID_EVAL_POINTS]

grid_results = []
for k, setting in enumerate(GRID):
    name = f"grid_r{setting['r']}_lr{setting['learning_rate']:g}"
    if name in CACHE["grid"]:
        print(name, "(cached)"); grid_results.append(CACHE["grid"][name]); continue
    CACHE["eval"].pop(name, None)   # an evaluation without its training run cannot be reused
    model, info = train_lora(setting, grid_train, 1, name)
    metrics = evaluate(model, name, grid_points, grid_val)
    grid_results.append(dict(name=name, **setting, **metrics, train_minutes=info["train_minutes"],
                             trainable_params=info["trainable_params"], history=info["history"]))
    CACHE["grid"][name] = grid_results[-1]; save_cache()
    BASE = release_adapter(model); del model; free()
    if DEVICE == "cuda":
        print(f"GPU memory still allocated after {name}: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

# the baseline on the grid subset, for the selection rule (the shared base model has no adapter at this point)
grid_baseline = evaluate(BASE, "baseline_grid_subset", grid_points, grid_val)

ok = [g for g in grid_results if g["leak_rate"] <= grid_baseline["leak_rate"]] or grid_results
best = min(ok, key=lambda g: g["val_loss"])
RESULTS["grid"] = [{k: v for k, v in g.items() if k != "history"} for g in grid_results]
RESULTS["grid_baseline"] = grid_baseline
RESULTS["best_setting"] = {k: best[k] for k in ["name", "r", "lora_alpha", "learning_rate"]}
pd.DataFrame(RESULTS["grid"]).set_index("name")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for g in grid_results:
    ax[0].plot([h["step"] for h in g["history"]], [h["loss"] for h in g["history"]], label=g["name"])
ax[0].set(title="Training loss by setting", xlabel="step", ylabel="loss"); ax[0].legend(fontsize=8)
names = [g["name"] for g in grid_results]
ax[1].bar(names, [g["val_loss"] for g in grid_results], color="#4C72B0")
ax[1].axhline(grid_baseline["val_loss"], color="grey", ls="--", label="baseline")
ax[1].set(title="Validation loss (grid subset)"); ax[1].legend(); ax[1].tick_params(axis="x", rotation=30)
x = np.arange(len(names))
ax[2].bar(x - 0.2, [g["leak_rate"] for g in grid_results], 0.4, label="leak rate", color="#C44E52")
ax[2].bar(x + 0.2, [g["question_rate"] for g in grid_results], 0.4, label="question rate", color="#55A868")
ax[2].axhline(grid_baseline["leak_rate"], color="#C44E52", ls="--", lw=1)
ax[2].axhline(grid_baseline["question_rate"], color="#55A868", ls="--", lw=1)
ax[2].set_xticks(x, names, rotation=30); ax[2].set(title="Reply metrics (dashed = baseline)"); ax[2].legend()
fig.tight_layout(); fig.savefig(OUT / "figures/02_grid_search.png"); plt.show()
print("best setting:", RESULTS["best_setting"])

## 7. Final training and comparison with the baseline

The best setting is trained on the full candidate train set and scored on all evaluation contexts.
The LoRA adapter is saved to the output folder.

In [ ]:
setting = {k: best[k] for k in ["r", "lora_alpha", "learning_rate"]}
CACHE["eval"].pop("tuned", None)   # always score the model trained in this run
model, info = train_lora(setting, train_enc, FINAL_EPOCHS, "final")
RESULTS["final_training"] = {k: v for k, v in info.items() if k != "history"}
RESULTS["tuned"] = evaluate(model, "tuned")
model.save_pretrained(OUT / "lora_adapter")
tok.save_pretrained(OUT / "lora_adapter")
final_history = info["history"]
BASE = release_adapter(model); del model, BASE; free()

compare = pd.DataFrame({name: RESULTS[name] for name in [baseline_name, "baseline_fp16", "tuned", "real_tutor_reference"] if name in RESULTS}).T
compare[["leak_rate", "question_rate", "rougeL", "bleu", "words", "val_loss"]]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot([h["step"] for h in final_history], [h["loss"] for h in final_history], color="#4C72B0")
ax[0].set(title=f"Final training loss ({RESULTS['best_setting']['name']})", xlabel="step", ylabel="loss")
cols = ["leak_rate", "question_rate", "rougeL", "bleu"]
rows = [n for n in [baseline_name, "baseline_fp16", "tuned", "real_tutor_reference"] if n in RESULTS]
x = np.arange(len(cols))
for k, n in enumerate(rows):
    ax[1].bar(x + (k - (len(rows) - 1) / 2) * 0.2, [RESULTS[n][c] for c in cols], 0.2, label=n)
ax[1].set_xticks(x, cols); ax[1].set(title="Baseline vs tuned on the same contexts"); ax[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figures/03_baseline_vs_tuned.png"); plt.show()

if (OUT / f"generations_{baseline_name}.csv").exists():   # missing when the baseline came from an uploaded cache
    side = pd.read_csv(OUT / f"generations_{baseline_name}.csv").merge(
        pd.read_csv(OUT / "generations_tuned.csv")[["dialogue_id", "reply"]], on="dialogue_id", suffixes=("_baseline", "_tuned"))
    display(side.head(5))

In [ ]:
json.dump(RESULTS, open(OUT / "results.json", "w"), indent=1, default=str)
print("saved to", OUT, sorted(p.name for p in OUT.iterdir()))

## Notes

- Unsloth's Qwen3.5 guide advises against 4-bit QLoRA for Qwen3.5 because quantization error is higher than usual for this model family, and recommends 16-bit LoRA instead. The fp16 baseline in section 5 shows how much the 4-bit model loses before any training.
- Without `flash-linear-attention`, transformers runs Qwen3.5's linear-attention layers in plain PyTorch, which is correct but much slower. If the first model call fails with a Triton error, set `FAST_KERNELS = False`, restart the session and run again.
- The T4 has no bf16 support, so everything runs in fp16. Watch for `nan` losses or empty replies (reported as `empty_replies`).
- The leak check only covers single-number final answers that are not already in the problem. Question rate counts question marks, so it is a rough proxy for Socratic style.
- ROUGE-L and BLEU compare the reply with one real tutor turn. A good Socratic reply can differ from it, so these scores are best read next to the leak and question rates.
- The MathDial test split and the 161 team scenarios are not used here; they are kept for the final evaluation.